In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import optuna
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr

optuna.logging.set_verbosity(optuna.logging.WARNING)

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]

# ── Optuna config ─────────────────────────────────────────────────────────────
# Each trial runs a 3-fold stratified CV (faster than 5-fold).
# Budget: 15h. Each trial ≈ 3-4 min → ~150 trials max. Use 80. N-Trials = 5 for now


In [ ]:
def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df_full = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df_full = df_full.dropna(subset=["target"])
    all_moons_full = np.array(sorted(df_full["moon"].unique()))

    def run_cv(window, recency_str, n_est, lr, num_leaves,
               subsample, colsample, min_child):
        window_moons = all_moons_full[-window:].tolist()
        df           = df_full[df_full["moon"].isin(window_moons)].copy()
        all_moons    = np.array(sorted(df["moon"].unique()))
        n_moons      = len(all_moons)
        moon_to_rank = {m: i for i, m in enumerate(all_moons)}

        N_QUINTILES = 3
        quintile    = (np.arange(n_moons) / n_moons * N_QUINTILES).astype(int)
        quintile    = np.clip(quintile, 0, N_QUINTILES - 1)

        fold_ics = []
        for fold in range(3):
            train_idx, val_idx = [], []
            for q in range(N_QUINTILES):
                q_idx  = np.where(quintile == q)[0]
                n_val  = max(1, len(q_idx) // 3)
                start  = (fold * n_val) % len(q_idx)
                v_pos  = np.arange(start, start + n_val) % len(q_idx)
                v_mask = np.zeros(len(q_idx), dtype=bool)
                v_mask[v_pos] = True
                val_idx.extend(q_idx[v_mask].tolist())
                train_idx.extend(q_idx[~v_mask].tolist())

            fold_tr  = all_moons[train_idx].tolist()
            fold_val = all_moons[val_idx].tolist()

            df_tr  = df[df["moon"].isin(fold_tr)].copy()
            df_val = df[df["moon"].isin(fold_val)].copy()

            ranks      = df_tr["moon"].map(moon_to_rank).values
            norm_ranks = ranks / max(ranks.max(), 1)
            weights    = np.exp(norm_ranks * recency_str)

            m = LGBMRegressor(
                n_estimators=n_est, learning_rate=lr,
                num_leaves=num_leaves, subsample=subsample,
                colsample_bytree=colsample,
                min_child_samples=min_child,
                random_state=fold, n_jobs=-1, verbose=-1,
            )
            m.fit(df_tr[feature_cols].fillna(0), df_tr["target"],
                  sample_weight=weights)

            corrs = []
            for moon in fold_val:
                grp = df_val[df_val["moon"] == moon]
                if len(grp) < 2: continue
                r, _ = pearsonr(
                    m.predict(grp[feature_cols].fillna(0)), grp["target"])
                if not np.isnan(r): corrs.append(r)
            fold_ics.append(float(np.mean(corrs)) if corrs else 0.0)

        return float(np.mean(fold_ics))

    def objective(trial):
        window      = trial.suggest_int("window", 200, 700)
        recency_str = trial.suggest_float("recency_str", 1.0, 10.0)
        n_est       = trial.suggest_int("n_estimators", 100, 600)
        lr          = trial.suggest_float("learning_rate", 0.01, 0.15, log=True)
        num_leaves  = trial.suggest_int("num_leaves", 15, 127)
        subsample   = trial.suggest_float("subsample", 0.5, 1.0)
        colsample   = trial.suggest_float("colsample_bytree", 0.4, 1.0)
        min_child   = trial.suggest_int("min_child_samples", 20, 200)
        return run_cv(window, recency_str, n_est, lr,
                      num_leaves, subsample, colsample, min_child)

    print(f"  Optuna: {5} trials, {3}-fold CV per trial")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=5, show_progress_bar=False)

    best = study.best_params
    best_ic = study.best_value
    print(f"  Best CV IC   = {best_ic:+.5f}")
    print(f"  Best params  = {best}")

    # ── Retrain 5-fold ensemble with best params on all data ──────────────────
    window_moons = all_moons_full[-best["window"]:].tolist()
    df           = df_full[df_full["moon"].isin(window_moons)].copy()
    all_moons    = np.array(sorted(df["moon"].unique()))
    n_moons      = len(all_moons)
    moon_to_rank = {m: i for i, m in enumerate(all_moons)}

    K           = 5
    N_QUINTILES = 3
    quintile    = (np.arange(n_moons) / n_moons * N_QUINTILES).astype(int)
    quintile    = np.clip(quintile, 0, N_QUINTILES - 1)

    models = []
    for fold in range(K):
        train_idx, val_idx = [], []
        for q in range(N_QUINTILES):
            q_idx  = np.where(quintile == q)[0]
            n_val  = max(1, len(q_idx) // K)
            start  = (fold * n_val) % len(q_idx)
            v_pos  = np.arange(start, start + n_val) % len(q_idx)
            v_mask = np.zeros(len(q_idx), dtype=bool)
            v_mask[v_pos] = True
            train_idx.extend(q_idx[~v_mask].tolist())

        fold_tr_moons = all_moons[train_idx].tolist()
        df_tr         = df[df["moon"].isin(fold_tr_moons)].copy()

        ranks      = df_tr["moon"].map(moon_to_rank).values
        norm_ranks = ranks / max(ranks.max(), 1)
        weights    = np.exp(norm_ranks * best["recency_str"])

        m = LGBMRegressor(
            n_estimators   = best["n_estimators"],
            learning_rate  = best["learning_rate"],
            num_leaves     = best["num_leaves"],
            subsample      = best["subsample"],
            colsample_bytree = best["colsample_bytree"],
            min_child_samples = best["min_child_samples"],
            random_state   = fold, n_jobs=-1, verbose=-1,
        )
        m.fit(df_tr[feature_cols].fillna(0), df_tr["target"],
              sample_weight=weights)
        models.append(m)
        print(f"    Final fold {fold+1}/{K} trained.")

    joblib.dump({
        "models": models, "features": feature_cols,
        "best_params": best, "best_cv_ic": best_ic,
    }, os.path.join(model_directory_path, "model.joblib"))
    print("  Saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    models   = obj["models"]
    features = obj["features"]
    X        = X_test[features].fillna(0)
    preds    = np.mean([m.predict(X) for m in models], axis=0)

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": preds,
    })


In [ ]:
import os
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

obj = joblib.load(os.path.join(model_dir, "model.joblib"))
print(f"\nBest params : {obj['best_params']}")
print(f"Best CV IC  : {obj['best_cv_ic']:.4f}\n")

results = []
for moon in splits["reduced_cloud"]:
    pred = infer(X_test_cloud[X_test_cloud["moon"] == moon],
                 model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)
corrs = []
for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon], on=["moon", "id"])
    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"Moon {moon}: Pearson r = {r:.4f}")
print(f"\nMean IC = {np.mean(corrs):.4f}")
